# Pose Landmarks Detection with MediaPipe Tasks

This notebook shows you how to use MediaPipe Tasks Python API to detect pose landmarks from images.

## Preparation

Let's start with installing MediaPipe.  
Then download an off-the-shelf model bundle. Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/pose_landmarker#models) for more information about this model bundle.

In [1]:
!pip install -q mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 5.5 MB/s eta 0:00:00


In [2]:
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

In [3]:
from google.colab import drive
from google.colab.patches import cv2_imshow
import cv2

drive.mount('/content/drive')

Mounted at /content/drive


## Visualization utilities

In [4]:
import numpy as np
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision


def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=3, circle_radius=4)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

## 🖼️ Get LandMarks for Image

In [ ]:
img = cv2.imread("/content/drive/MyDrive/Dance App/dance_pose_sample.png")

numpy_image = img.copy()

if numpy_image.shape[2] == 4:
    numpy_image = cv2.cvtColor(numpy_image, cv2.COLOR_RGBA2RGB)

## Running inference and visualizing the results

The final step is to run pose landmark detection on your selected image. This involves creating your PoseLandmarker object, loading your image, running detection, and finally, the optional step of displaying the image with visualizations.

Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/pose_landmarker/python) to learn more about configuration options that this solution supports.


In [ ]:
# STEP 1: Import the necessary modules.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# STEP 2: Create an PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)
detector = vision.PoseLandmarker.create_from_options(options)

# STEP 3: Load the input image.
image = mp.Image.create_from_file("/content/drive/MyDrive/Dance App/dance_pose_sample.png")

# STEP 4: Detect pose landmarks from the input image.
detection_result = detector.detect(image)

# STEP 5: Process the detection result. In this case, visualize it.
annotated_image = draw_landmarks_on_image(numpy_image, detection_result)

# 4. Display the image
cv2_imshow(cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))

## 📽️ Get Landmarks for video

In [19]:
# =============================================================================
# Mask background + mirror + z-inversion + anatomically-stable skeleton
# -----------------------------------------------------------------------------
# 1. Never freeze a joint while a pose exists — always take MediaPipe's current
#    estimate for all 33 joints (unsure ones are still body-coherent).
# 2. Smooth each joint with an EMA whose weight is the model's confidence, so
#    unsure joints lean on previous frames instead of jumping or freezing.
# 3. Learn each bone's length from confident frames; if a bone touching an
#    unsure joint stretches past MAX_STRETCH x that length, pull the unsure
#    joint back along the bone so proportions stay human.
# 4. Only carry positions forward unchanged when NO pose is detected at all.
# Topology always comes from vision.PoseLandmarksConnections.POSE_LANDMARKS.
# =============================================================================
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

INPUT_VIDEO       = "/content/drive/MyDrive/Dance App/dance_sample_0_6_seconds.mp4"
OUTPUT_VIDEO_MASK = "/content/drive/MyDrive/Dance App/output/dance_sample_0_6_seconds_mask_mirror.mp4"

# ---- Config ------------------------------------------------------------------
POSE_CONNECTIONS = vision.PoseLandmarksConnections.POSE_LANDMARKS
NUM_LANDMARKS    = 33
CONF_THRESHOLD   = 0.5     # below this a joint counts as "unsure" (MediaPipe's default)
ALPHA_MIN        = 0.20    # EMA weight for a totally unsure joint (leans on previous frames)
ALPHA_MAX        = 0.90    # EMA weight for a fully confident joint (tracks almost 1:1)
MAX_STRETCH      = 1.50    # a bone with an unsure joint may not exceed this x its learned length
STALE_LIMIT      = None    # frames to keep drawing a joint that hasn't been confident (None = forever)
MIRROR           = True
INVERT_Z         = True

BONE_COLOR     = (0, 255, 0)      # both joints confident
UNSURE_COLOR   = (255, 165, 0)    # at least one joint below CONF_THRESHOLD (set = BONE_COLOR to hide)
BONE_THICK     = 3
JOINT_STYLE    = drawing_styles.get_default_pose_landmarks_style()
JOINT_FALLBACK = drawing_utils.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2)


def _ends(conn):
    return (conn.start, conn.end) if hasattr(conn, "start") else (conn[0], conn[1])


class SkeletonTracker:
    def __init__(self, width, height):
        self.w, self.h = width, height
        self.px      = np.full((NUM_LANDMARKS, 2), np.nan, dtype=np.float32)  # smoothed pixel positions
        self.z       = np.full(NUM_LANDMARKS, np.nan, dtype=np.float32)
        self.conf    = np.zeros(NUM_LANDMARKS, dtype=np.float32)
        self.age     = np.zeros(NUM_LANDMARKS, dtype=np.int32)                 # frames since last confident
        self.ref_len = {}                                                       # (s, e) -> learned bone length

    @staticmethod
    def _conf(lm):
        v = 1.0 if lm.visibility is None else lm.visibility
        p = 1.0 if lm.presence   is None else lm.presence
        return float(min(v, p))

    def _to_px(self, lm):
        x = 1.0 - lm.x if MIRROR else lm.x
        return np.array([x * (self.w - 1), lm.y * (self.h - 1)], dtype=np.float32)

    def update(self, landmarks):
        """landmarks: the 33 NormalizedLandmarks of one pose, or None if nobody was detected."""
        if landmarks is None:
            self.age += 1                      # nothing new: keep last positions, just mark older
            return

        for i in range(NUM_LANDMARKS):
            lm     = landmarks[i]
            c      = self._conf(lm)
            target = self._to_px(lm)
            zval   = -lm.z if INVERT_Z else lm.z

            if np.isnan(self.px[i, 0]):        # first sighting: take as-is
                self.px[i] = target
                self.z[i]  = zval
            else:                              # confidence-weighted EMA
                a = float(np.clip(c, ALPHA_MIN, ALPHA_MAX))
                self.px[i] = a * target + (1.0 - a) * self.px[i]
                self.z[i]  = a * zval   + (1.0 - a) * self.z[i]

            self.conf[i] = c
            self.age[i]  = 0 if c >= CONF_THRESHOLD else self.age[i] + 1

        self._enforce_bone_lengths()

    def _enforce_bone_lengths(self):
        for conn in POSE_CONNECTIONS:
            s, e = _ends(conn)
            if np.isnan(self.px[s, 0]) or np.isnan(self.px[e, 0]):
                continue
            d = self.px[e] - self.px[s]
            L = float(np.hypot(d[0], d[1]))
            if L < 1e-3:
                continue
            key = (s, e)
            if self.conf[s] >= CONF_THRESHOLD and self.conf[e] >= CONF_THRESHOLD:
                # learn / slowly adapt the reference length from confident frames only
                self.ref_len[key] = L if key not in self.ref_len else 0.9 * self.ref_len[key] + 0.1 * L
            elif key in self.ref_len and L > MAX_STRETCH * self.ref_len[key]:
                # pull the LESS confident joint back along the bone to the learned length
                ref = self.ref_len[key]
                if self.conf[s] >= self.conf[e]:
                    self.px[e] = self.px[s] + d * (ref / L)
                else:
                    self.px[s] = self.px[e] - d * (ref / L)

    def _drawable(self, i):
        if np.isnan(self.px[i, 0]):
            return False
        return STALE_LIMIT is None or self.age[i] <= STALE_LIMIT

    def _pt(self, i):
        return (int(round(float(self.px[i, 0]))), int(round(float(self.px[i, 1]))))

    def draw(self, image):
        for conn in POSE_CONNECTIONS:
            s, e = _ends(conn)
            if self._drawable(s) and self._drawable(e):
                unsure = self.conf[s] < CONF_THRESHOLD or self.conf[e] < CONF_THRESHOLD
                cv2.line(image, self._pt(s), self._pt(e), UNSURE_COLOR if unsure else BONE_COLOR, BONE_THICK)
        for i in range(NUM_LANDMARKS):
            if self._drawable(i):
                spec = JOINT_STYLE.get(i, JOINT_FALLBACK)
                cv2.circle(image, self._pt(i), spec.circle_radius, spec.color, spec.thickness)
        return image


# ---- Detector ----------------------------------------------------------------
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True,
    running_mode=vision.RunningMode.VIDEO,
)
detector = vision.PoseLandmarker.create_from_options(options)

# ---- Video I/O ---------------------------------------------------------------
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"fps={fps:.3f}  size={width}x{height}  frames={total_frames}")

writer  = cv2.VideoWriter(OUTPUT_VIDEO_MASK, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
tracker = SkeletonTracker(width, height)
last_bg = np.zeros((height, width, 3), dtype=np.uint8)

# ---- Main loop ---------------------------------------------------------------
frame_idx = 0
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    frame_rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image     = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    timestamp_ms = int(round(frame_idx * 1000.0 / fps))
    result       = detector.detect_for_video(mp_image, timestamp_ms)

    # 1) Background: segmentation mask as 3-channel grayscale (carry last one forward if missing)
    if result.segmentation_masks:
        seg     = np.squeeze(result.segmentation_masks[0].numpy_view())
        bg      = (np.clip(seg, 0.0, 1.0) * 255).astype(np.uint8)
        last_bg = np.stack([bg] * 3, axis=-1)
    bg = cv2.flip(last_bg, 1) if MIRROR else last_bg.copy()

    # 2) Feed this frame's joints into the tracker (None if nobody detected)
    tracker.update(result.pose_landmarks[0] if result.pose_landmarks else None)

    # 3) Draw every bone in POSE_CONNECTIONS + every joint
    frame_out = tracker.draw(bg)

    writer.write(cv2.cvtColor(frame_out, cv2.COLOR_RGB2BGR))

    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"processed {frame_idx}/{total_frames}")

cap.release()
writer.release()
detector.close()
print(f"Done. Wrote {frame_idx} frames to {OUTPUT_VIDEO_MASK}")

fps=30.000  size=1080x1920  frames=179
processed 30/179
processed 60/179
processed 90/179
processed 120/179
processed 150/179
Done. Wrote 179 frames to /content/drive/MyDrive/Dance App/output/dance_sample_0_6_seconds_mask_mirror.mp4
